In [2]:
import pandas as pd
import numpy as np
import os

# =========================
# Load cleaned files
# =========================

train_path = "../MLOpsedian/data/processed/train_cleaned.csv"
test_path = "../MLOpsedian/data/processed/test_cleaned.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

id_col = "record_id"
target = "flood_risk_score"

print("Original train shape:", train.shape)
print("Original test shape:", test.shape)

print("\nOriginal missing values:")
print("Train missing:", train.isnull().sum().sum())
print("Test missing:", test.isnull().sum().sum())

print("\nRows with any missing in train:", train.isnull().any(axis=1).sum())
print("Rows without missing in train:", (~train.isnull().any(axis=1)).sum())


# =========================
# VERSION 1: Drop missing rows only
# =========================

train_dropmissing = train.dropna().copy()
test_dropmissing = test.copy()

print("\n==============================")
print("VERSION 1: DROP MISSING ONLY")
print("==============================")
print("Train shape:", train_dropmissing.shape)
print("Test shape:", test_dropmissing.shape)
print("Train missing:", train_dropmissing.isnull().sum().sum())
print("Test missing:", test_dropmissing.isnull().sum().sum())
print("Duplicate train IDs:", train_dropmissing[id_col].duplicated().sum())
print("Duplicate test IDs:", test_dropmissing[id_col].duplicated().sum())
print("Train-only columns:", set(train_dropmissing.columns) - set(test_dropmissing.columns))
print("Test-only columns:", set(test_dropmissing.columns) - set(train_dropmissing.columns))


# =========================
# VERSION 2: Drop missing + invalid rows
# =========================

invalid_rules = {
    "distance_to_river_m": {"min": 0, "max": None},
    "rainfall_7d_mm": {"min": 0, "max": None},
    "monthly_rainfall_mm": {"min": 0, "max": None},
    "built_up_percent": {"min": 0, "max": 100},
    "drainage_index": {"min": 0, "max": 1},
    "ndvi": {"min": -1, "max": 1},
    "ndwi": {"min": -1, "max": 1},
    "elevation_m": {"min": 0, "max": None}
}

missing_mask = train.isnull().any(axis=1)
invalid_mask = pd.Series(False, index=train.index)

invalid_report = []

for col, rule in invalid_rules.items():
    if col in train.columns:
        col_invalid = pd.Series(False, index=train.index)
        
        if rule["min"] is not None:
            col_invalid |= train[col] < rule["min"]
        
        if rule["max"] is not None:
            col_invalid |= train[col] > rule["max"]
        
        col_invalid = col_invalid.fillna(False)
        invalid_mask |= col_invalid
        
        invalid_report.append({
            "column": col,
            "invalid_count": int(col_invalid.sum()),
            "min_value": train[col].min(),
            "max_value": train[col].max()
        })

invalid_report_df = pd.DataFrame(invalid_report)

train_dropmissing_invalid = train.loc[~(missing_mask | invalid_mask)].copy()
test_dropmissing_invalid = test.copy()

print("\n==============================")
print("VERSION 2: DROP MISSING + INVALID")
print("==============================")
print("Rows with missing:", int(missing_mask.sum()))
print("Rows with invalid values:", int(invalid_mask.sum()))
print("Rows with missing OR invalid:", int((missing_mask | invalid_mask).sum()))
print("Rows kept:", train_dropmissing_invalid.shape[0])

print("\nTrain shape:", train_dropmissing_invalid.shape)
print("Test shape:", test_dropmissing_invalid.shape)
print("Train missing:", train_dropmissing_invalid.isnull().sum().sum())
print("Test missing:", test_dropmissing_invalid.isnull().sum().sum())
print("Duplicate train IDs:", train_dropmissing_invalid[id_col].duplicated().sum())
print("Duplicate test IDs:", test_dropmissing_invalid[id_col].duplicated().sum())
print("Train-only columns:", set(train_dropmissing_invalid.columns) - set(test_dropmissing_invalid.columns))
print("Test-only columns:", set(test_dropmissing_invalid.columns) - set(train_dropmissing_invalid.columns))

print("\nInvalid value report:")
display(invalid_report_df)


# =========================
# Check target balance
# =========================

def target_balance(df, name):
    bins = pd.cut(
        df[target],
        bins=[-0.001, 0.2, 0.4, 0.6, 0.8, 1.001],
        labels=[
            "0.0-0.2 very low",
            "0.2-0.4 low",
            "0.4-0.6 medium",
            "0.6-0.8 high",
            "0.8-1.0 very high"
        ]
    )
    
    report = pd.DataFrame({
        "count": bins.value_counts().sort_index(),
        "percentage": (bins.value_counts(normalize=True).sort_index() * 100).round(2)
    })
    
    print("\n==============================")
    print(name)
    print("==============================")
    print("Shape:", df.shape)
    print(report)
    
    return report

balance_full = target_balance(train, "Original cleaned train")
balance_dropmissing = target_balance(train_dropmissing, "Drop missing train")
balance_strict = target_balance(train_dropmissing_invalid, "Drop missing + invalid train")


# =========================
# Save files
# =========================

os.makedirs("../MLOpsedian/data/processed", exist_ok=True)
os.makedirs("../MLOpsedian/reports", exist_ok=True)

train_dropmissing.to_csv("../MLOpsedian/data/processed/train_dropmissing.csv", index=False)
test_dropmissing.to_csv("../MLOpsedian/data/processed/test_dropmissing.csv", index=False)

train_dropmissing_invalid.to_csv("../MLOpsedian/data/processed/train_dropmissing_invalid.csv", index=False)
test_dropmissing_invalid.to_csv("../MLOpsedian/data/processed/test_dropmissing_invalid.csv", index=False)

invalid_report_df.to_csv("../MLOpsedian/reports/invalid_rows_removed_report.csv", index=False)
balance_full.to_csv("../MLOpsedian/reports/balance_original_cleaned_train.csv")
balance_dropmissing.to_csv("../MLOpsedian/reports/balance_dropmissing_train.csv")
balance_strict.to_csv("../MLOpsedian/reports/balance_dropmissing_invalid_train.csv")

print("\n==============================")
print("FILES SAVED")
print("==============================")
print("../MLOpsedian/data/processed/train_dropmissing.csv")
print("../MLOpsedian/data/processed/test_dropmissing.csv")
print("../MLOpsedian/data/processed/train_dropmissing_invalid.csv")
print("../MLOpsedian/data/processed/test_dropmissing_invalid.csv")
print("../MLOpsedian/reports/invalid_rows_removed_report.csv")

Original train shape: (19700, 69)
Original test shape: (5300, 68)

Original missing values:
Train missing: 32761
Test missing: 0

Rows with any missing in train: 4182
Rows without missing in train: 15518

VERSION 1: DROP MISSING ONLY
Train shape: (15518, 69)
Test shape: (5300, 68)
Train missing: 0
Test missing: 0
Duplicate train IDs: 0
Duplicate test IDs: 0
Train-only columns: {'flood_risk_score'}
Test-only columns: set()

VERSION 2: DROP MISSING + INVALID
Rows with missing: 4182
Rows with invalid values: 683
Rows with missing OR invalid: 4779
Rows kept: 14921

Train shape: (14921, 69)
Test shape: (5300, 68)
Train missing: 0
Test missing: 0
Duplicate train IDs: 0
Duplicate test IDs: 0
Train-only columns: {'flood_risk_score'}
Test-only columns: set()

Invalid value report:


,column,invalid_count,min_value,max_value
0,distance_to_river_m,117,-485.397750,16802.000000
1,rainfall_7d_mm,96,-49.199358,914.823318
2,monthly_rainfall_mm,96,-79.170571,2032.274155
3,built_up_percent,72,1.000000,178.263721
4,drainage_index,71,0.003000,1.779877
5,ndvi,71,-0.792000,1.595039
6,ndwi,71,-1.000000,1.596308
7,elevation_m,97,-79.564149,2148.000000



Original cleaned train
Shape: (19700, 69)
                   count  percentage
flood_risk_score                    
0.0-0.2 very low    2390       12.13
0.2-0.4 low         4402       22.35
0.4-0.6 medium      7650       38.83
0.6-0.8 high        3353       17.02
0.8-1.0 very high   1905        9.67

Drop missing train
Shape: (15518, 69)
                   count  percentage
flood_risk_score                    
0.0-0.2 very low    1899       12.24
0.2-0.4 low         3455       22.26
0.4-0.6 medium      5993       38.62
0.6-0.8 high        2655       17.11
0.8-1.0 very high   1516        9.77

Drop missing + invalid train
Shape: (14921, 69)
                   count  percentage
flood_risk_score                    
0.0-0.2 very low    1816       12.17
0.2-0.4 low         3338       22.37
0.4-0.6 medium      5757       38.58
0.6-0.8 high        2561       17.16
0.8-1.0 very high   1449        9.71

FILES SAVED
../MLOpsedian/data/processed/train_dropmissing.csv
../MLOpsedian/data/processed

In [3]:
# ============================================================
# CatBoost Alpha Pack on Drop-Missing Dataset
# One-cell version
# ============================================================

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

# ============================================================
# 1. Load drop-missing datasets
# ============================================================

train_path = "../MLOpsedian/data/processed/train_dropmissing.csv"
test_path = "../MLOpsedian/data/processed/test_dropmissing.csv"

# Backup path in case your files are saved inside MLOpsedian folder


train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

id_col = "record_id"
target = "flood_risk_score"

print("Train path:", train_path)
print("Test path:", test_path)
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train missing:", train.isnull().sum().sum())
print("Test missing:", test.isnull().sum().sum())
print("Duplicate train IDs:", train[id_col].duplicated().sum())
print("Duplicate test IDs:", test[id_col].duplicated().sum())
print("Train-only columns:", set(train.columns) - set(test.columns))
print("Test-only columns:", set(test.columns) - set(train.columns))
print("Target valid:", train[target].between(0, 1).all())

# ============================================================
# 2. Add Alpha Pack engineered features
# ============================================================

def add_alpha_pack_features(df):
    df = df.copy()
    eps = 1e-5
    
    # Use clipped columns if available
    if "distance_to_river_m_clipped" in df.columns:
        distance = df["distance_to_river_m_clipped"]
    else:
        distance = df["distance_to_river_m"].clip(lower=0)
    
    if "rainfall_7d_mm_clipped" in df.columns:
        rainfall_7d = df["rainfall_7d_mm_clipped"]
    else:
        rainfall_7d = df["rainfall_7d_mm"].clip(lower=0)
    
    inundation = df["inundation_area_sqm"].clip(lower=0)
    
    # 1. Golden distance-rainfall ratio
    df["GOLDEN_distance_rainfall_ratio"] = (
        np.log1p(distance) - np.log1p(rainfall_7d + eps)
    )
    
    # 2. Flood spread ratio
    df["distance_to_river_DIV_inundation_area"] = (
        distance / (inundation + eps)
    )
    
    # 3. Water velocity proxy
    df["distance_to_river_DIV_rainfall_7d"] = (
        distance / (rainfall_7d + eps)
    )
    
    # 4. Water volume proxy
    df["rainfall_7d_MULT_inundation_area"] = (
        rainfall_7d * inundation
    )
    
    new_cols = [
        "GOLDEN_distance_rainfall_ratio",
        "distance_to_river_DIV_inundation_area",
        "distance_to_river_DIV_rainfall_7d",
        "rainfall_7d_MULT_inundation_area"
    ]
    
    for col in new_cols:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)
        df[col] = df[col].fillna(df[col].median())
    
    return df

train_fe = add_alpha_pack_features(train)
test_fe = add_alpha_pack_features(test)

print("\nAfter feature engineering:")
print("Train shape:", train_fe.shape)
print("Test shape:", test_fe.shape)
print("Train missing:", train_fe.isnull().sum().sum())
print("Test missing:", test_fe.isnull().sum().sum())

# ============================================================
# 3. Define Alpha Pack features
# ============================================================

alpha_pack_features = [
    "district",
    "distance_to_river_DIV_inundation_area",
    "distance_to_river_DIV_rainfall_7d",
    "GOLDEN_distance_rainfall_ratio",
    "generation_date",
    "reason_not_good_to_live",
    "inundation_area_sqm",
    "infrastructure_score",
    "extreme_weather_index",
    "terrain_roughness_index",
    "road_quality",
    "monthly_rainfall_mm_log1p",
    "landcover",
    "rainfall_7d_MULT_inundation_area",
    "place_name",
    "rainfall_7d_mm",
    "seasonal_index",
    "ndwi_qmap",
    "latitude",
    "longitude",
    "distance_to_river_m",
    "ndwi",
    "water_supply",
    "nearest_evac_km_log1p",
    "elevation_m_yeojohnson",
    "monthly_rainfall_mm",
    "socioeconomic_status_index",
    "population_density_per_km2_log1p",
    "water_presence_flag",
    "nearest_hospital_km_log1p",
    "ndvi_qmap",
    "flood_occurrence_current_event",
    "rainfall_7d_mm_log1p",
    "soil_type",
    "population_density_per_km2"
]

missing_train_features = [col for col in alpha_pack_features if col not in train_fe.columns]
missing_test_features = [col for col in alpha_pack_features if col not in test_fe.columns]

print("\nFeature check:")
print("Number of alpha pack features:", len(alpha_pack_features))
print("Missing train features:", missing_train_features)
print("Missing test features:", missing_test_features)

if len(missing_train_features) > 0 or len(missing_test_features) > 0:
    raise ValueError("Some alpha pack features are missing. Stop and check columns.")

# ============================================================
# 4. Prepare X, y, X_test
# ============================================================

X = train_fe[alpha_pack_features].copy()
y = train_fe[target].copy()

X_test = test_fe[alpha_pack_features].copy()
test_ids = test_fe[id_col].copy()

print("\nModel data:")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)
print("Missing in X:", X.isnull().sum().sum())
print("Missing in X_test:", X_test.isnull().sum().sum())
print("Feature columns match:", list(X.columns) == list(X_test.columns))

# ============================================================
# 5. Identify categorical features
# ============================================================

cat_features = [
    col for col in X.columns
    if X[col].dtype == "object" or str(X[col].dtype) == "str"
]

print("\nCategorical features:")
print("Number of categorical features:", len(cat_features))
print(cat_features)

# ============================================================
# 6. Target bins for balance-aware StratifiedKFold
# ============================================================

target_bins = pd.cut(
    y,
    bins=[-0.001, 0.2, 0.4, 0.6, 0.8, 1.001],
    labels=[0, 1, 2, 3, 4]
)

print("\nTarget bin distribution:")
print(target_bins.value_counts().sort_index())

# ============================================================
# 7. Submission validation function
# ============================================================

def validate_submission(submission):
    print("\nSubmission shape:", submission.shape)
    print(submission.head())
    
    print("\nMissing values:")
    print(submission.isnull().sum())
    
    print("\nPrediction range:")
    print("Min:", submission["flood_risk_score"].min())
    print("Max:", submission["flood_risk_score"].max())
    
    assert submission.shape[0] == len(test_ids)
    assert list(submission.columns) == ["record_id", "flood_risk_score"]
    assert submission["flood_risk_score"].isnull().sum() == 0
    assert submission["flood_risk_score"].between(0, 1).all()
    
    print("\nSubmission is valid.")

# ============================================================
# 8. Train CatBoost Alpha Pack on drop-missing data
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rmse_scores = []
mae_scores = []
r2_scores = []

test_preds = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, target_bins), 1):
    print("\n" + "=" * 70)
    print(f"Fold {fold}")
    
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    
    model = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.03,
        depth=6,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42 + fold,
        verbose=200,
        early_stopping_rounds=200,
        allow_writing_files=False
    )
    
    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )
    
    valid_preds = model.predict(X_valid)
    valid_preds = np.clip(valid_preds, 0, 1)
    
    rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))
    mae = mean_absolute_error(y_valid, valid_preds)
    r2 = r2_score(y_valid, valid_preds)
    
    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)
    
    print("Fold RMSE:", rmse)
    print("Fold MAE :", mae)
    print("Fold R²  :", r2)
    
    fold_test_preds = model.predict(X_test)
    test_preds += fold_test_preds / skf.n_splits

print("\n" + "=" * 70)
print("CatBoost Alpha Pack Drop-Missing CV Results")
print("Average RMSE:", np.mean(rmse_scores))
print("Average MAE :", np.mean(mae_scores))
print("Average R²  :", np.mean(r2_scores))

# ============================================================
# 9. Create submission
# ============================================================

dropmissing_preds = np.clip(test_preds, 0, 1)

print("\nPrediction distribution:")
print("Prediction min:", dropmissing_preds.min())
print("Prediction max:", dropmissing_preds.max())
print("Prediction mean:", dropmissing_preds.mean())
print("Prediction std:", dropmissing_preds.std())

sub_dropmissing = pd.DataFrame({
    "record_id": test_ids,
    "flood_risk_score": dropmissing_preds
})

validate_submission(sub_dropmissing)

os.makedirs("../submissions", exist_ok=True)

sub_dropmissing.to_csv(
    "../submissions/sub_v010_catboost_alpha_pack_dropmissing.csv",
    index=False
)

print("\nSaved: ../submissions/sub_v010_catboost_alpha_pack_dropmissing.csv")

# ============================================================
# 10. Compare with previous best
# ============================================================

comparison = pd.DataFrame([
    {
        "version": "sub_v008_catboost_alpha_pack",
        "dataset": "Full cleaned train",
        "rows": 19700,
        "cv_rmse": 0.23164778640435246,
        "cv_mae": 0.17673736118136243,
        "cv_r2": 0.03463008143352864,
        "public_score": 0.38308
    },
    {
        "version": "sub_v010_catboost_alpha_pack_dropmissing",
        "dataset": "Drop missing train",
        "rows": len(train),
        "cv_rmse": np.mean(rmse_scores),
        "cv_mae": np.mean(mae_scores),
        "cv_r2": np.mean(r2_scores),
        "public_score": None
    }
])

comparison["rmse_vs_previous"] = comparison["cv_rmse"] - comparison.loc[0, "cv_rmse"]
comparison["mae_vs_previous"] = comparison["cv_mae"] - comparison.loc[0, "cv_mae"]
comparison["r2_vs_previous"] = comparison["cv_r2"] - comparison.loc[0, "cv_r2"]

os.makedirs("../reports", exist_ok=True)

comparison.to_csv(
    "../reports/catboost_alpha_pack_dropmissing_comparison.csv",
    index=False
)

print("\nComparison with previous best:")
display(comparison)

Train path: ../MLOpsedian/data/processed/train_dropmissing.csv
Test path: ../MLOpsedian/data/processed/test_dropmissing.csv
Train shape: (15518, 69)
Test shape: (5300, 68)
Train missing: 0
Test missing: 0
Duplicate train IDs: 0
Duplicate test IDs: 0
Train-only columns: {'flood_risk_score'}
Test-only columns: set()
Target valid: True

After feature engineering:
Train shape: (15518, 73)
Test shape: (5300, 72)
Train missing: 0
Test missing: 0

Feature check:
Number of alpha pack features: 35
Missing train features: []
Missing test features: []

Model data:
X shape: (15518, 35)
y shape: (15518,)
X_test shape: (5300, 35)
Missing in X: 0
Missing in X_test: 0
Feature columns match: True

Categorical features:
Number of categorical features: 10
['district', 'generation_date', 'reason_not_good_to_live', 'road_quality', 'landcover', 'place_name', 'water_supply', 'water_presence_flag', 'flood_occurrence_current_event', 'soil_type']

Target bin distribution:
flood_risk_score
0    1899
1    3455
2 

,version,dataset,rows,cv_rmse,cv_mae,cv_r2,public_score,rmse_vs_previous,mae_vs_previous,r2_vs_previous
0,sub_v008_catboost_alpha_pack,Full cleaned train,19700,0.231648,0.176737,0.034630,0.38308,0.000000,0.000000,0.000000
1,sub_v010_catboost_alpha_pack_dropmissing,Drop missing train,15518,0.231805,0.176606,0.040233,NaN,0.000157,-0.000132,0.005603


In [4]:
# ============================================================
# CatBoost Alpha Pack on Drop-Missing + Invalid-Removed Dataset
# One-cell version
# ============================================================

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

# ============================================================
# 1. Load strict-clean datasets
# ============================================================

train_path = "../MLOpsedian/data/processed/train_dropmissing_invalid.csv"
test_path = "../MLOpsedian/data/processed/test_dropmissing_invalid.csv"



train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

id_col = "record_id"
target = "flood_risk_score"

print("Train path:", train_path)
print("Test path:", test_path)
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train missing:", train.isnull().sum().sum())
print("Test missing:", test.isnull().sum().sum())
print("Duplicate train IDs:", train[id_col].duplicated().sum())
print("Duplicate test IDs:", test[id_col].duplicated().sum())
print("Train-only columns:", set(train.columns) - set(test.columns))
print("Test-only columns:", set(test.columns) - set(train.columns))
print("Target valid:", train[target].between(0, 1).all())

# ============================================================
# 2. Add Alpha Pack engineered features
# ============================================================

def add_alpha_pack_features(df):
    df = df.copy()
    eps = 1e-5
    
    if "distance_to_river_m_clipped" in df.columns:
        distance = df["distance_to_river_m_clipped"]
    else:
        distance = df["distance_to_river_m"].clip(lower=0)
    
    if "rainfall_7d_mm_clipped" in df.columns:
        rainfall_7d = df["rainfall_7d_mm_clipped"]
    else:
        rainfall_7d = df["rainfall_7d_mm"].clip(lower=0)
    
    inundation = df["inundation_area_sqm"].clip(lower=0)
    
    df["GOLDEN_distance_rainfall_ratio"] = (
        np.log1p(distance) - np.log1p(rainfall_7d + eps)
    )
    
    df["distance_to_river_DIV_inundation_area"] = (
        distance / (inundation + eps)
    )
    
    df["distance_to_river_DIV_rainfall_7d"] = (
        distance / (rainfall_7d + eps)
    )
    
    df["rainfall_7d_MULT_inundation_area"] = (
        rainfall_7d * inundation
    )
    
    new_cols = [
        "GOLDEN_distance_rainfall_ratio",
        "distance_to_river_DIV_inundation_area",
        "distance_to_river_DIV_rainfall_7d",
        "rainfall_7d_MULT_inundation_area"
    ]
    
    for col in new_cols:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)
        df[col] = df[col].fillna(df[col].median())
    
    return df

train_fe = add_alpha_pack_features(train)
test_fe = add_alpha_pack_features(test)

print("\nAfter feature engineering:")
print("Train shape:", train_fe.shape)
print("Test shape:", test_fe.shape)
print("Train missing:", train_fe.isnull().sum().sum())
print("Test missing:", test_fe.isnull().sum().sum())

# ============================================================
# 3. Define Alpha Pack features
# ============================================================

alpha_pack_features = [
    "district",
    "distance_to_river_DIV_inundation_area",
    "distance_to_river_DIV_rainfall_7d",
    "GOLDEN_distance_rainfall_ratio",
    "generation_date",
    "reason_not_good_to_live",
    "inundation_area_sqm",
    "infrastructure_score",
    "extreme_weather_index",
    "terrain_roughness_index",
    "road_quality",
    "monthly_rainfall_mm_log1p",
    "landcover",
    "rainfall_7d_MULT_inundation_area",
    "place_name",
    "rainfall_7d_mm",
    "seasonal_index",
    "ndwi_qmap",
    "latitude",
    "longitude",
    "distance_to_river_m",
    "ndwi",
    "water_supply",
    "nearest_evac_km_log1p",
    "elevation_m_yeojohnson",
    "monthly_rainfall_mm",
    "socioeconomic_status_index",
    "population_density_per_km2_log1p",
    "water_presence_flag",
    "nearest_hospital_km_log1p",
    "ndvi_qmap",
    "flood_occurrence_current_event",
    "rainfall_7d_mm_log1p",
    "soil_type",
    "population_density_per_km2"
]

missing_train_features = [col for col in alpha_pack_features if col not in train_fe.columns]
missing_test_features = [col for col in alpha_pack_features if col not in test_fe.columns]

print("\nFeature check:")
print("Number of alpha pack features:", len(alpha_pack_features))
print("Missing train features:", missing_train_features)
print("Missing test features:", missing_test_features)

if len(missing_train_features) > 0 or len(missing_test_features) > 0:
    raise ValueError("Some alpha pack features are missing. Stop and check columns.")

# ============================================================
# 4. Prepare X, y, X_test
# ============================================================

X = train_fe[alpha_pack_features].copy()
y = train_fe[target].copy()

X_test = test_fe[alpha_pack_features].copy()
test_ids = test_fe[id_col].copy()

print("\nModel data:")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)
print("Missing in X:", X.isnull().sum().sum())
print("Missing in X_test:", X_test.isnull().sum().sum())
print("Feature columns match:", list(X.columns) == list(X_test.columns))

# ============================================================
# 5. Identify categorical features
# ============================================================

cat_features = [
    col for col in X.columns
    if X[col].dtype == "object" or str(X[col].dtype) == "str"
]

print("\nCategorical features:")
print("Number of categorical features:", len(cat_features))
print(cat_features)

# ============================================================
# 6. Target bins for balance-aware StratifiedKFold
# ============================================================

target_bins = pd.cut(
    y,
    bins=[-0.001, 0.2, 0.4, 0.6, 0.8, 1.001],
    labels=[0, 1, 2, 3, 4]
)

print("\nTarget bin distribution:")
print(target_bins.value_counts().sort_index())

# ============================================================
# 7. Submission validation function
# ============================================================

def validate_submission(submission):
    print("\nSubmission shape:", submission.shape)
    print(submission.head())
    
    print("\nMissing values:")
    print(submission.isnull().sum())
    
    print("\nPrediction range:")
    print("Min:", submission["flood_risk_score"].min())
    print("Max:", submission["flood_risk_score"].max())
    
    assert submission.shape[0] == len(test_ids)
    assert list(submission.columns) == ["record_id", "flood_risk_score"]
    assert submission["flood_risk_score"].isnull().sum() == 0
    assert submission["flood_risk_score"].between(0, 1).all()
    
    print("\nSubmission is valid.")

# ============================================================
# 8. Train CatBoost Alpha Pack on strict-clean data
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rmse_scores = []
mae_scores = []
r2_scores = []

test_preds = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, target_bins), 1):
    print("\n" + "=" * 70)
    print(f"Fold {fold}")
    
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    
    model = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.03,
        depth=6,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42 + fold,
        verbose=200,
        early_stopping_rounds=200,
        allow_writing_files=False
    )
    
    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )
    
    valid_preds = model.predict(X_valid)
    valid_preds = np.clip(valid_preds, 0, 1)
    
    rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))
    mae = mean_absolute_error(y_valid, valid_preds)
    r2 = r2_score(y_valid, valid_preds)
    
    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)
    
    print("Fold RMSE:", rmse)
    print("Fold MAE :", mae)
    print("Fold R²  :", r2)
    
    fold_test_preds = model.predict(X_test)
    test_preds += fold_test_preds / skf.n_splits

print("\n" + "=" * 70)
print("CatBoost Alpha Pack Drop-Missing + Invalid CV Results")
print("Average RMSE:", np.mean(rmse_scores))
print("Average MAE :", np.mean(mae_scores))
print("Average R²  :", np.mean(r2_scores))

# ============================================================
# 9. Create submission
# ============================================================

strict_preds = np.clip(test_preds, 0, 1)

print("\nPrediction distribution:")
print("Prediction min:", strict_preds.min())
print("Prediction max:", strict_preds.max())
print("Prediction mean:", strict_preds.mean())
print("Prediction std:", strict_preds.std())

sub_strict = pd.DataFrame({
    "record_id": test_ids,
    "flood_risk_score": strict_preds
})

validate_submission(sub_strict)

os.makedirs("../submissions", exist_ok=True)

sub_strict.to_csv(
    "../submissions/sub_v011_catboost_alpha_pack_dropmissing_invalid.csv",
    index=False
)

print("\nSaved: ../submissions/sub_v011_catboost_alpha_pack_dropmissing_invalid.csv")

# ============================================================
# 10. Compare with previous best versions
# ============================================================

comparison = pd.DataFrame([
    {
        "version": "sub_v008_catboost_alpha_pack",
        "dataset": "Full cleaned train",
        "rows": 19700,
        "cv_rmse": 0.23164778640435246,
        "cv_mae": 0.17673736118136243,
        "cv_r2": 0.03463008143352864,
        "public_score": 0.38308
    },
    {
        "version": "sub_v010_catboost_alpha_pack_dropmissing",
        "dataset": "Drop missing train",
        "rows": 15518,
        "cv_rmse": 0.23180475883494558,
        "cv_mae": 0.17660570642311946,
        "cv_r2": 0.04023286825885215,
        "public_score": 0.38176
    },
    {
        "version": "sub_v011_catboost_alpha_pack_dropmissing_invalid",
        "dataset": "Drop missing + invalid train",
        "rows": len(train),
        "cv_rmse": np.mean(rmse_scores),
        "cv_mae": np.mean(mae_scores),
        "cv_r2": np.mean(r2_scores),
        "public_score": None
    }
])

comparison["rmse_vs_best_public"] = comparison["cv_rmse"] - comparison.loc[1, "cv_rmse"]
comparison["mae_vs_best_public"] = comparison["cv_mae"] - comparison.loc[1, "cv_mae"]
comparison["r2_vs_best_public"] = comparison["cv_r2"] - comparison.loc[1, "cv_r2"]

os.makedirs("../reports", exist_ok=True)

comparison.to_csv(
    "../reports/catboost_alpha_pack_strict_clean_comparison.csv",
    index=False
)

print("\nComparison with previous versions:")
display(comparison)

Train path: ../MLOpsedian/data/processed/train_dropmissing_invalid.csv
Test path: ../MLOpsedian/data/processed/test_dropmissing_invalid.csv
Train shape: (14921, 69)
Test shape: (5300, 68)
Train missing: 0
Test missing: 0
Duplicate train IDs: 0
Duplicate test IDs: 0
Train-only columns: {'flood_risk_score'}
Test-only columns: set()
Target valid: True

After feature engineering:
Train shape: (14921, 73)
Test shape: (5300, 72)
Train missing: 0
Test missing: 0

Feature check:
Number of alpha pack features: 35
Missing train features: []
Missing test features: []

Model data:
X shape: (14921, 35)
y shape: (14921,)
X_test shape: (5300, 35)
Missing in X: 0
Missing in X_test: 0
Feature columns match: True

Categorical features:
Number of categorical features: 10
['district', 'generation_date', 'reason_not_good_to_live', 'road_quality', 'landcover', 'place_name', 'water_supply', 'water_presence_flag', 'flood_occurrence_current_event', 'soil_type']

Target bin distribution:
flood_risk_score
0    1

,version,dataset,rows,cv_rmse,cv_mae,cv_r2,public_score,rmse_vs_best_public,mae_vs_best_public,r2_vs_best_public
0,sub_v008_catboost_alpha_pack,Full cleaned train,19700,0.231648,0.176737,0.034630,0.38308,-0.000157,0.000132,-0.005603
1,sub_v010_catboost_alpha_pack_dropmissing,Drop missing train,15518,0.231805,0.176606,0.040233,0.38176,0.000000,0.000000,0.000000
2,sub_v011_catboost_alpha_pack_dropmissing_invalid,Drop missing + invalid train,14921,0.231628,0.176713,0.039624,NaN,-0.000176,0.000108,-0.000609


In [5]:
# ============================================================
# V012: CatBoost Full + Alpha Features on Drop-Missing Dataset
# ============================================================

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

# ============================================================
# 1. Load drop-missing datasets
# ============================================================

train_path = "../data/processed/train_dropmissing.csv"
test_path = "../data/processed/test_dropmissing.csv"

# Backup path if your project folder is one level different
if not os.path.exists(train_path):
    train_path = "../MLOpsedian/data/processed/train_dropmissing.csv"
    test_path = "../MLOpsedian/data/processed/test_dropmissing.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

id_col = "record_id"
target = "flood_risk_score"

print("Train path:", train_path)
print("Test path:", test_path)
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train missing:", train.isnull().sum().sum())
print("Test missing:", test.isnull().sum().sum())
print("Duplicate train IDs:", train[id_col].duplicated().sum())
print("Duplicate test IDs:", test[id_col].duplicated().sum())
print("Train-only columns:", set(train.columns) - set(test.columns))
print("Test-only columns:", set(test.columns) - set(train.columns))
print("Target valid:", train[target].between(0, 1).all())

# ============================================================
# 2. Add Alpha Pack engineered features
# ============================================================

def add_alpha_pack_features(df):
    df = df.copy()
    eps = 1e-5
    
    if "distance_to_river_m_clipped" in df.columns:
        distance = df["distance_to_river_m_clipped"]
    else:
        distance = df["distance_to_river_m"].clip(lower=0)
    
    if "rainfall_7d_mm_clipped" in df.columns:
        rainfall_7d = df["rainfall_7d_mm_clipped"]
    else:
        rainfall_7d = df["rainfall_7d_mm"].clip(lower=0)
    
    inundation = df["inundation_area_sqm"].clip(lower=0)
    
    df["GOLDEN_distance_rainfall_ratio"] = (
        np.log1p(distance) - np.log1p(rainfall_7d + eps)
    )
    
    df["distance_to_river_DIV_inundation_area"] = (
        distance / (inundation + eps)
    )
    
    df["distance_to_river_DIV_rainfall_7d"] = (
        distance / (rainfall_7d + eps)
    )
    
    df["rainfall_7d_MULT_inundation_area"] = (
        rainfall_7d * inundation
    )
    
    new_cols = [
        "GOLDEN_distance_rainfall_ratio",
        "distance_to_river_DIV_inundation_area",
        "distance_to_river_DIV_rainfall_7d",
        "rainfall_7d_MULT_inundation_area"
    ]
    
    for col in new_cols:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)
        df[col] = df[col].fillna(df[col].median())
    
    return df

train_fe = add_alpha_pack_features(train)
test_fe = add_alpha_pack_features(test)

print("\nAfter feature engineering:")
print("Train shape:", train_fe.shape)
print("Test shape:", test_fe.shape)
print("Train missing:", train_fe.isnull().sum().sum())
print("Test missing:", test_fe.isnull().sum().sum())

# ============================================================
# 3. Use Full Features + Alpha Features
# ============================================================

base_drop_cols = [
    id_col,
    target,
    "generation_date_missing"
]

# Drop any feature that is constant in train because it gives no learning signal
constant_train_cols = [
    col for col in train_fe.columns
    if col not in base_drop_cols and train_fe[col].nunique(dropna=False) <= 1
]

drop_cols = base_drop_cols + constant_train_cols

feature_cols = [
    col for col in train_fe.columns
    if col not in drop_cols
]

print("\nDropped columns:")
print(drop_cols)

print("\nNumber of features used:", len(feature_cols))
print("Constant columns dropped:", constant_train_cols)

# Check feature alignment
missing_test_features = [col for col in feature_cols if col not in test_fe.columns]
print("Missing test features:", missing_test_features)

if len(missing_test_features) > 0:
    raise ValueError("Some train features are missing in test. Stop and check.")

# ============================================================
# 4. Prepare X, y, X_test
# ============================================================

X = train_fe[feature_cols].copy()
y = train_fe[target].copy()

X_test = test_fe[feature_cols].copy()
test_ids = test_fe[id_col].copy()

print("\nModel data:")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)
print("Missing in X:", X.isnull().sum().sum())
print("Missing in X_test:", X_test.isnull().sum().sum())
print("Feature columns match:", list(X.columns) == list(X_test.columns))

# ============================================================
# 5. Identify categorical features
# ============================================================

cat_features = [
    col for col in X.columns
    if X[col].dtype == "object" or str(X[col].dtype) == "str"
]

print("\nCategorical features:")
print("Number of categorical features:", len(cat_features))
print(cat_features)

# ============================================================
# 6. Target bins for balance-aware StratifiedKFold
# ============================================================

target_bins = pd.cut(
    y,
    bins=[-0.001, 0.2, 0.4, 0.6, 0.8, 1.001],
    labels=[0, 1, 2, 3, 4]
)

print("\nTarget bin distribution:")
print(target_bins.value_counts().sort_index())

# ============================================================
# 7. Submission validation function
# ============================================================

def validate_submission(submission):
    print("\nSubmission shape:", submission.shape)
    print(submission.head())
    
    print("\nMissing values:")
    print(submission.isnull().sum())
    
    print("\nPrediction range:")
    print("Min:", submission["flood_risk_score"].min())
    print("Max:", submission["flood_risk_score"].max())
    
    assert submission.shape[0] == len(test_ids)
    assert list(submission.columns) == ["record_id", "flood_risk_score"]
    assert submission["flood_risk_score"].isnull().sum() == 0
    assert submission["flood_risk_score"].between(0, 1).all()
    
    print("\nSubmission is valid.")

# ============================================================
# 8. Train CatBoost Full + Alpha on drop-missing data
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rmse_scores = []
mae_scores = []
r2_scores = []

test_preds = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, target_bins), 1):
    print("\n" + "=" * 70)
    print(f"Fold {fold}")
    
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    
    model = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.03,
        depth=6,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42 + fold,
        verbose=200,
        early_stopping_rounds=200,
        allow_writing_files=False
    )
    
    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )
    
    valid_preds = model.predict(X_valid)
    valid_preds = np.clip(valid_preds, 0, 1)
    
    rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))
    mae = mean_absolute_error(y_valid, valid_preds)
    r2 = r2_score(y_valid, valid_preds)
    
    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)
    
    print("Fold RMSE:", rmse)
    print("Fold MAE :", mae)
    print("Fold R²  :", r2)
    
    fold_test_preds = model.predict(X_test)
    test_preds += fold_test_preds / skf.n_splits

print("\n" + "=" * 70)
print("CatBoost Full + Alpha Drop-Missing CV Results")
print("Average RMSE:", np.mean(rmse_scores))
print("Average MAE :", np.mean(mae_scores))
print("Average R²  :", np.mean(r2_scores))

# ============================================================
# 9. Create submission
# ============================================================

full_alpha_preds = np.clip(test_preds, 0, 1)

print("\nPrediction distribution:")
print("Prediction min:", full_alpha_preds.min())
print("Prediction max:", full_alpha_preds.max())
print("Prediction mean:", full_alpha_preds.mean())
print("Prediction std:", full_alpha_preds.std())

sub_full_alpha = pd.DataFrame({
    "record_id": test_ids,
    "flood_risk_score": full_alpha_preds
})

validate_submission(sub_full_alpha)

os.makedirs("../submissions", exist_ok=True)

sub_full_alpha.to_csv(
    "../submissions/sub_v012_catboost_full_plus_alpha_dropmissing.csv",
    index=False
)

print("\nSaved: ../submissions/sub_v012_catboost_full_plus_alpha_dropmissing.csv")

# ============================================================
# 10. Compare with previous versions
# ============================================================

comparison = pd.DataFrame([
    {
        "version": "sub_v008_catboost_alpha_pack",
        "dataset": "Full cleaned train",
        "features": "35 Alpha Pack",
        "rows": 19700,
        "cv_rmse": 0.23164778640435246,
        "cv_mae": 0.17673736118136243,
        "cv_r2": 0.03463008143352864,
        "prediction_std": 0.042242,
        "public_score": 0.38308
    },
    {
        "version": "sub_v010_catboost_alpha_pack_dropmissing",
        "dataset": "Drop missing train",
        "features": "35 Alpha Pack",
        "rows": 15518,
        "cv_rmse": 0.23180475883494558,
        "cv_mae": 0.17660570642311946,
        "cv_r2": 0.04023286825885215,
        "prediction_std": 0.044751843087494254,
        "public_score": 0.38176
    },
    {
        "version": "sub_v011_catboost_alpha_pack_dropmissing_invalid",
        "dataset": "Drop missing + invalid train",
        "features": "35 Alpha Pack",
        "rows": 14921,
        "cv_rmse": 0.23162834341472624,
        "cv_mae": 0.1767132396544392,
        "cv_r2": 0.03962422453198171,
        "prediction_std": 0.04174568759804983,
        "public_score": 0.38309
    },
    {
        "version": "sub_v012_catboost_full_plus_alpha_dropmissing",
        "dataset": "Drop missing train",
        "features": f"Full + Alpha ({len(feature_cols)} features)",
        "rows": len(train),
        "cv_rmse": np.mean(rmse_scores),
        "cv_mae": np.mean(mae_scores),
        "cv_r2": np.mean(r2_scores),
        "prediction_std": full_alpha_preds.std(),
        "public_score": None
    }
])

comparison["rmse_vs_best_public"] = comparison["cv_rmse"] - comparison.loc[1, "cv_rmse"]
comparison["mae_vs_best_public"] = comparison["cv_mae"] - comparison.loc[1, "cv_mae"]
comparison["r2_vs_best_public"] = comparison["cv_r2"] - comparison.loc[1, "cv_r2"]
comparison["pred_std_vs_best_public"] = comparison["prediction_std"] - comparison.loc[1, "prediction_std"]

os.makedirs("../reports", exist_ok=True)

comparison.to_csv(
    "../reports/catboost_full_plus_alpha_dropmissing_comparison.csv",
    index=False
)

print("\nComparison with previous versions:")
display(comparison)

Train path: ../MLOpsedian/data/processed/train_dropmissing.csv
Test path: ../MLOpsedian/data/processed/test_dropmissing.csv
Train shape: (15518, 69)
Test shape: (5300, 68)
Train missing: 0
Test missing: 0
Duplicate train IDs: 0
Duplicate test IDs: 0
Train-only columns: {'flood_risk_score'}
Test-only columns: set()
Target valid: True

After feature engineering:
Train shape: (15518, 73)
Test shape: (5300, 72)
Train missing: 0
Test missing: 0

Dropped columns:
['record_id', 'flood_risk_score', 'generation_date_missing']

Number of features used: 70
Constant columns dropped: []
Missing test features: []

Model data:
X shape: (15518, 70)
y shape: (15518,)
X_test shape: (5300, 70)
Missing in X: 0
Missing in X_test: 0
Feature columns match: True

Categorical features:
Number of categorical features: 14
['district', 'place_name', 'landcover', 'soil_type', 'water_supply', 'electricity', 'road_quality', 'urban_rural', 'water_presence_flag', 'flood_occurrence_current_event', 'is_good_to_live', 'r

,version,dataset,features,rows,cv_rmse,cv_mae,cv_r2,prediction_std,public_score,rmse_vs_best_public,mae_vs_best_public,r2_vs_best_public,pred_std_vs_best_public
0,sub_v008_catboost_alpha_pack,Full cleaned train,35 Alpha Pack,19700,0.231648,0.176737,0.034630,0.042242,0.38308,-0.000157,0.000132,-0.005603,-0.002510
1,sub_v010_catboost_alpha_pack_dropmissing,Drop missing train,35 Alpha Pack,15518,0.231805,0.176606,0.040233,0.044752,0.38176,0.000000,0.000000,0.000000,0.000000
2,sub_v011_catboost_alpha_pack_dropmissing_invalid,Drop missing + invalid train,35 Alpha Pack,14921,0.231628,0.176713,0.039624,0.041746,0.38309,-0.000176,0.000108,-0.000609,-0.003006
3,sub_v012_catboost_full_plus_alpha_dropmissing,Drop missing train,Full + Alpha (70 features),15518,0.231948,0.176718,0.039049,0.043601,NaN,0.000144,0.000113,-0.001184,-0.001151


In [6]:
# ============================================================
# V013: Correlation-Guided CatBoost on Drop-Missing Dataset
# Uses 35 Alpha Pack + selected extra correlation-supported features
# ============================================================

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

# ============================================================
# 1. Load drop-missing datasets
# ============================================================

train_path = "../data/processed/train_dropmissing.csv"
test_path = "../data/processed/test_dropmissing.csv"

if not os.path.exists(train_path):
    train_path = "../MLOpsedian/data/processed/train_dropmissing.csv"
    test_path = "../MLOpsedian/data/processed/test_dropmissing.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

id_col = "record_id"
target = "flood_risk_score"

print("Train path:", train_path)
print("Test path:", test_path)
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train missing:", train.isnull().sum().sum())
print("Test missing:", test.isnull().sum().sum())
print("Duplicate train IDs:", train[id_col].duplicated().sum())
print("Duplicate test IDs:", test[id_col].duplicated().sum())
print("Train-only columns:", set(train.columns) - set(test.columns))
print("Test-only columns:", set(test.columns) - set(train.columns))
print("Target valid:", train[target].between(0, 1).all())

# ============================================================
# 2. Add Alpha Pack engineered features
# ============================================================

def add_alpha_pack_features(df):
    df = df.copy()
    eps = 1e-5
    
    if "distance_to_river_m_clipped" in df.columns:
        distance = df["distance_to_river_m_clipped"]
    else:
        distance = df["distance_to_river_m"].clip(lower=0)
    
    if "rainfall_7d_mm_clipped" in df.columns:
        rainfall_7d = df["rainfall_7d_mm_clipped"]
    else:
        rainfall_7d = df["rainfall_7d_mm"].clip(lower=0)
    
    inundation = df["inundation_area_sqm"].clip(lower=0)
    
    df["GOLDEN_distance_rainfall_ratio"] = (
        np.log1p(distance) - np.log1p(rainfall_7d + eps)
    )
    
    df["distance_to_river_DIV_inundation_area"] = (
        distance / (inundation + eps)
    )
    
    df["distance_to_river_DIV_rainfall_7d"] = (
        distance / (rainfall_7d + eps)
    )
    
    df["rainfall_7d_MULT_inundation_area"] = (
        rainfall_7d * inundation
    )
    
    new_cols = [
        "GOLDEN_distance_rainfall_ratio",
        "distance_to_river_DIV_inundation_area",
        "distance_to_river_DIV_rainfall_7d",
        "rainfall_7d_MULT_inundation_area"
    ]
    
    for col in new_cols:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)
        df[col] = df[col].fillna(df[col].median())
    
    return df

train_fe = add_alpha_pack_features(train)
test_fe = add_alpha_pack_features(test)

print("\nAfter feature engineering:")
print("Train shape:", train_fe.shape)
print("Test shape:", test_fe.shape)
print("Train missing:", train_fe.isnull().sum().sum())
print("Test missing:", test_fe.isnull().sum().sum())

# ============================================================
# 3. V013 correlation-guided feature set
# ============================================================

v013_features = [
    "district",
    "distance_to_river_DIV_inundation_area",
    "distance_to_river_DIV_rainfall_7d",
    "GOLDEN_distance_rainfall_ratio",
    "generation_date",
    "reason_not_good_to_live",
    "inundation_area_sqm",
    "infrastructure_score",
    "extreme_weather_index",
    "terrain_roughness_index",
    "road_quality",
    "monthly_rainfall_mm_log1p",
    "landcover",
    "rainfall_7d_MULT_inundation_area",
    "place_name",
    "rainfall_7d_mm",
    "seasonal_index",
    "ndwi_qmap",
    "latitude",
    "longitude",
    "distance_to_river_m",
    "ndwi",
    "water_supply",
    "nearest_evac_km_log1p",
    "elevation_m_yeojohnson",
    "monthly_rainfall_mm",
    "socioeconomic_status_index",
    "population_density_per_km2_log1p",
    "water_presence_flag",
    "nearest_hospital_km_log1p",
    "ndvi_qmap",
    "flood_occurrence_current_event",
    "rainfall_7d_mm_log1p",
    "soil_type",
    "population_density_per_km2",
    
    # Extra correlation-supported features
    "distance_to_river_m_log1p",
    "distance_to_river_m_clipped",
    "rainfall_7d_mm_clipped",
    "historical_flood_count",
    "monthly_rainfall_mm_clipped",
    "ndwi_clipped",
    "built_up_percent_qmap",
    "built_up_percent",
    "built_up_percent_clipped",
    "drainage_index_yeojohnson",
    "drainage_index",
    "drainage_index_clipped",
    "generation_day",
    "is_good_to_live"
]

# Remove accidental duplicates while preserving order
v013_features = list(dict.fromkeys(v013_features))

missing_train_features = [col for col in v013_features if col not in train_fe.columns]
missing_test_features = [col for col in v013_features if col not in test_fe.columns]

print("\nFeature check:")
print("Number of V013 features:", len(v013_features))
print("Missing train features:", missing_train_features)
print("Missing test features:", missing_test_features)

if len(missing_train_features) > 0 or len(missing_test_features) > 0:
    raise ValueError("Some V013 features are missing. Stop and check columns.")

# ============================================================
# 4. Prepare model data
# ============================================================

X = train_fe[v013_features].copy()
y = train_fe[target].copy()

X_test = test_fe[v013_features].copy()
test_ids = test_fe[id_col].copy()

print("\nModel data:")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)
print("Missing in X:", X.isnull().sum().sum())
print("Missing in X_test:", X_test.isnull().sum().sum())
print("Feature columns match:", list(X.columns) == list(X_test.columns))

cat_features = [
    col for col in X.columns
    if X[col].dtype == "object" or str(X[col].dtype) == "str"
]

print("\nCategorical features:")
print("Number of categorical features:", len(cat_features))
print(cat_features)

# ============================================================
# 5. Target-binned StratifiedKFold
# ============================================================

target_bins = pd.cut(
    y,
    bins=[-0.001, 0.2, 0.4, 0.6, 0.8, 1.001],
    labels=[0, 1, 2, 3, 4]
)

print("\nTarget bin distribution:")
print(target_bins.value_counts().sort_index())

# ============================================================
# 6. Submission validation function
# ============================================================

def validate_submission(submission):
    print("\nSubmission shape:", submission.shape)
    print(submission.head())
    
    print("\nMissing values:")
    print(submission.isnull().sum())
    
    print("\nPrediction range:")
    print("Min:", submission["flood_risk_score"].min())
    print("Max:", submission["flood_risk_score"].max())
    
    assert submission.shape[0] == len(test_ids)
    assert list(submission.columns) == ["record_id", "flood_risk_score"]
    assert submission["flood_risk_score"].isnull().sum() == 0
    assert submission["flood_risk_score"].between(0, 1).all()
    
    print("\nSubmission is valid.")

# ============================================================
# 7. Train CatBoost V013
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rmse_scores = []
mae_scores = []
r2_scores = []

test_preds = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, target_bins), 1):
    print("\n" + "=" * 70)
    print(f"Fold {fold}")
    
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    
    model = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.03,
        depth=6,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42 + fold,
        verbose=200,
        early_stopping_rounds=200,
        allow_writing_files=False
    )
    
    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )
    
    valid_preds = model.predict(X_valid)
    valid_preds = np.clip(valid_preds, 0, 1)
    
    rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))
    mae = mean_absolute_error(y_valid, valid_preds)
    r2 = r2_score(y_valid, valid_preds)
    
    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)
    
    print("Fold RMSE:", rmse)
    print("Fold MAE :", mae)
    print("Fold R²  :", r2)
    
    fold_test_preds = model.predict(X_test)
    test_preds += fold_test_preds / skf.n_splits

print("\n" + "=" * 70)
print("V013 Correlation-Guided CatBoost CV Results")
print("Average RMSE:", np.mean(rmse_scores))
print("Average MAE :", np.mean(mae_scores))
print("Average R²  :", np.mean(r2_scores))

# ============================================================
# 8. Create submission
# ============================================================

v013_preds = np.clip(test_preds, 0, 1)

print("\nPrediction distribution:")
print("Prediction min:", v013_preds.min())
print("Prediction max:", v013_preds.max())
print("Prediction mean:", v013_preds.mean())
print("Prediction std:", v013_preds.std())

sub_v013 = pd.DataFrame({
    "record_id": test_ids,
    "flood_risk_score": v013_preds
})

validate_submission(sub_v013)

os.makedirs("../submissions", exist_ok=True)

sub_v013.to_csv(
    "../submissions/sub_v013_catboost_corr_guided_dropmissing.csv",
    index=False
)

print("\nSaved: ../submissions/sub_v013_catboost_corr_guided_dropmissing.csv")

# ============================================================
# 9. Compare with previous versions
# ============================================================

comparison = pd.DataFrame([
    {
        "version": "sub_v008_catboost_alpha_pack",
        "dataset": "Full cleaned train",
        "features": "35 Alpha Pack",
        "rows": 19700,
        "cv_rmse": 0.23164778640435246,
        "cv_mae": 0.17673736118136243,
        "cv_r2": 0.03463008143352864,
        "prediction_std": 0.042242,
        "public_score": 0.38308
    },
    {
        "version": "sub_v010_catboost_alpha_pack_dropmissing",
        "dataset": "Drop missing train",
        "features": "35 Alpha Pack",
        "rows": 15518,
        "cv_rmse": 0.23180475883494558,
        "cv_mae": 0.17660570642311946,
        "cv_r2": 0.04023286825885215,
        "prediction_std": 0.044751843087494254,
        "public_score": 0.38176
    },
    {
        "version": "sub_v012_catboost_full_plus_alpha_dropmissing",
        "dataset": "Drop missing train",
        "features": "Full + Alpha, 70 features",
        "rows": 15518,
        "cv_rmse": 0.2319484336072119,
        "cv_mae": 0.17671839229429026,
        "cv_r2": 0.03904856170750963,
        "prediction_std": 0.04360086243404623,
        "public_score": None
    },
    {
        "version": "sub_v013_catboost_corr_guided_dropmissing",
        "dataset": "Drop missing train",
        "features": f"Correlation-guided, {len(v013_features)} features",
        "rows": len(train),
        "cv_rmse": np.mean(rmse_scores),
        "cv_mae": np.mean(mae_scores),
        "cv_r2": np.mean(r2_scores),
        "prediction_std": v013_preds.std(),
        "public_score": None
    }
])

comparison["rmse_vs_best_public"] = comparison["cv_rmse"] - comparison.loc[1, "cv_rmse"]
comparison["mae_vs_best_public"] = comparison["cv_mae"] - comparison.loc[1, "cv_mae"]
comparison["r2_vs_best_public"] = comparison["cv_r2"] - comparison.loc[1, "cv_r2"]
comparison["pred_std_vs_best_public"] = comparison["prediction_std"] - comparison.loc[1, "prediction_std"]

os.makedirs("../reports", exist_ok=True)

comparison.to_csv(
    "../reports/catboost_v013_corr_guided_comparison.csv",
    index=False
)

print("\nComparison with previous versions:")
display(comparison)

Train path: ../MLOpsedian/data/processed/train_dropmissing.csv
Test path: ../MLOpsedian/data/processed/test_dropmissing.csv
Train shape: (15518, 69)
Test shape: (5300, 68)
Train missing: 0
Test missing: 0
Duplicate train IDs: 0
Duplicate test IDs: 0
Train-only columns: {'flood_risk_score'}
Test-only columns: set()
Target valid: True

After feature engineering:
Train shape: (15518, 73)
Test shape: (5300, 72)
Train missing: 0
Test missing: 0

Feature check:
Number of V013 features: 49
Missing train features: []
Missing test features: []

Model data:
X shape: (15518, 49)
y shape: (15518,)
X_test shape: (5300, 49)
Missing in X: 0
Missing in X_test: 0
Feature columns match: True

Categorical features:
Number of categorical features: 11
['district', 'generation_date', 'reason_not_good_to_live', 'road_quality', 'landcover', 'place_name', 'water_supply', 'water_presence_flag', 'flood_occurrence_current_event', 'soil_type', 'is_good_to_live']

Target bin distribution:
flood_risk_score
0    1899

,version,dataset,features,rows,cv_rmse,cv_mae,cv_r2,prediction_std,public_score,rmse_vs_best_public,mae_vs_best_public,r2_vs_best_public,pred_std_vs_best_public
0,sub_v008_catboost_alpha_pack,Full cleaned train,35 Alpha Pack,19700,0.231648,0.176737,0.034630,0.042242,0.38308,-0.000157,0.000132,-0.005603,-0.002510
1,sub_v010_catboost_alpha_pack_dropmissing,Drop missing train,35 Alpha Pack,15518,0.231805,0.176606,0.040233,0.044752,0.38176,0.000000,0.000000,0.000000,0.000000
2,sub_v012_catboost_full_plus_alpha_dropmissing,Drop missing train,"Full + Alpha, 70 features",15518,0.231948,0.176718,0.039049,0.043601,NaN,0.000144,0.000113,-0.001184,-0.001151
3,sub_v013_catboost_corr_guided_dropmissing,Drop missing train,"Correlation-guided, 49 features",15518,0.231866,0.176720,0.039736,0.043629,NaN,0.000061,0.000115,-0.000497,-0.001123


In [7]:
# ============================================================
# V014: CatBoost Alpha Pack Drop-Missing Seed Ensemble
# Uses the proven V010 setup + multiple random seeds
# ============================================================

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

# ============================================================
# 1. Load best dataset: drop-missing
# ============================================================

train_path = "../data/processed/train_dropmissing.csv"
test_path = "../data/processed/test_dropmissing.csv"

if not os.path.exists(train_path):
    train_path = "../MLOpsedian/data/processed/train_dropmissing.csv"
    test_path = "../MLOpsedian/data/processed/test_dropmissing.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

id_col = "record_id"
target = "flood_risk_score"

print("Train path:", train_path)
print("Test path:", test_path)
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train missing:", train.isnull().sum().sum())
print("Test missing:", test.isnull().sum().sum())
print("Duplicate train IDs:", train[id_col].duplicated().sum())
print("Duplicate test IDs:", test[id_col].duplicated().sum())
print("Train-only columns:", set(train.columns) - set(test.columns))
print("Test-only columns:", set(test.columns) - set(train.columns))
print("Target valid:", train[target].between(0, 1).all())

# ============================================================
# 2. Add Alpha Pack engineered features
# ============================================================

def add_alpha_pack_features(df):
    df = df.copy()
    eps = 1e-5
    
    if "distance_to_river_m_clipped" in df.columns:
        distance = df["distance_to_river_m_clipped"]
    else:
        distance = df["distance_to_river_m"].clip(lower=0)
    
    if "rainfall_7d_mm_clipped" in df.columns:
        rainfall_7d = df["rainfall_7d_mm_clipped"]
    else:
        rainfall_7d = df["rainfall_7d_mm"].clip(lower=0)
    
    inundation = df["inundation_area_sqm"].clip(lower=0)
    
    df["GOLDEN_distance_rainfall_ratio"] = (
        np.log1p(distance) - np.log1p(rainfall_7d + eps)
    )
    
    df["distance_to_river_DIV_inundation_area"] = (
        distance / (inundation + eps)
    )
    
    df["distance_to_river_DIV_rainfall_7d"] = (
        distance / (rainfall_7d + eps)
    )
    
    df["rainfall_7d_MULT_inundation_area"] = (
        rainfall_7d * inundation
    )
    
    new_cols = [
        "GOLDEN_distance_rainfall_ratio",
        "distance_to_river_DIV_inundation_area",
        "distance_to_river_DIV_rainfall_7d",
        "rainfall_7d_MULT_inundation_area"
    ]
    
    for col in new_cols:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)
        df[col] = df[col].fillna(df[col].median())
    
    return df

train_fe = add_alpha_pack_features(train)
test_fe = add_alpha_pack_features(test)

print("\nAfter feature engineering:")
print("Train shape:", train_fe.shape)
print("Test shape:", test_fe.shape)
print("Train missing:", train_fe.isnull().sum().sum())
print("Test missing:", test_fe.isnull().sum().sum())

# ============================================================
# 3. Use exact V010 Alpha Pack feature set
# ============================================================

alpha_pack_features = [
    "district",
    "distance_to_river_DIV_inundation_area",
    "distance_to_river_DIV_rainfall_7d",
    "GOLDEN_distance_rainfall_ratio",
    "generation_date",
    "reason_not_good_to_live",
    "inundation_area_sqm",
    "infrastructure_score",
    "extreme_weather_index",
    "terrain_roughness_index",
    "road_quality",
    "monthly_rainfall_mm_log1p",
    "landcover",
    "rainfall_7d_MULT_inundation_area",
    "place_name",
    "rainfall_7d_mm",
    "seasonal_index",
    "ndwi_qmap",
    "latitude",
    "longitude",
    "distance_to_river_m",
    "ndwi",
    "water_supply",
    "nearest_evac_km_log1p",
    "elevation_m_yeojohnson",
    "monthly_rainfall_mm",
    "socioeconomic_status_index",
    "population_density_per_km2_log1p",
    "water_presence_flag",
    "nearest_hospital_km_log1p",
    "ndvi_qmap",
    "flood_occurrence_current_event",
    "rainfall_7d_mm_log1p",
    "soil_type",
    "population_density_per_km2"
]

missing_train_features = [col for col in alpha_pack_features if col not in train_fe.columns]
missing_test_features = [col for col in alpha_pack_features if col not in test_fe.columns]

print("\nFeature check:")
print("Number of Alpha Pack features:", len(alpha_pack_features))
print("Missing train features:", missing_train_features)
print("Missing test features:", missing_test_features)

if len(missing_train_features) > 0 or len(missing_test_features) > 0:
    raise ValueError("Some Alpha Pack features are missing. Stop and check columns.")

# ============================================================
# 4. Prepare model data
# ============================================================

X = train_fe[alpha_pack_features].copy()
y = train_fe[target].copy()

X_test = test_fe[alpha_pack_features].copy()
test_ids = test_fe[id_col].copy()

print("\nModel data:")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)
print("Missing in X:", X.isnull().sum().sum())
print("Missing in X_test:", X_test.isnull().sum().sum())
print("Feature columns match:", list(X.columns) == list(X_test.columns))

cat_features = [
    col for col in X.columns
    if X[col].dtype == "object" or str(X[col].dtype) == "str"
]

print("\nCategorical features:")
print("Number of categorical features:", len(cat_features))
print(cat_features)

# ============================================================
# 5. Target bins for balance-aware folds
# ============================================================

target_bins = pd.cut(
    y,
    bins=[-0.001, 0.2, 0.4, 0.6, 0.8, 1.001],
    labels=[0, 1, 2, 3, 4]
)

print("\nTarget bin distribution:")
print(target_bins.value_counts().sort_index())

# ============================================================
# 6. Submission validation function
# ============================================================

def validate_submission(submission):
    print("\nSubmission shape:", submission.shape)
    print(submission.head())
    
    print("\nMissing values:")
    print(submission.isnull().sum())
    
    print("\nPrediction range:")
    print("Min:", submission["flood_risk_score"].min())
    print("Max:", submission["flood_risk_score"].max())
    
    assert submission.shape[0] == len(test_ids)
    assert list(submission.columns) == ["record_id", "flood_risk_score"]
    assert submission["flood_risk_score"].isnull().sum() == 0
    assert submission["flood_risk_score"].between(0, 1).all()
    
    print("\nSubmission is valid.")

# ============================================================
# 7. Seed ensemble training
# ============================================================

seeds = [42, 202, 777, 2026, 9999]
n_splits = 5

all_seed_oof_preds = []
all_seed_test_preds = []
seed_score_rows = []

for seed in seeds:
    print("\n" + "#" * 80)
    print(f"Training seed: {seed}")
    print("#" * 80)
    
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed
    )
    
    seed_oof_preds = np.zeros(len(X))
    seed_test_preds = np.zeros(len(X_test))
    
    fold_rmse_scores = []
    fold_mae_scores = []
    fold_r2_scores = []
    
    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, target_bins), 1):
        print("\n" + "=" * 70)
        print(f"Seed {seed} | Fold {fold}")
        
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        
        model = CatBoostRegressor(
            iterations=2000,
            learning_rate=0.03,
            depth=6,
            loss_function="RMSE",
            eval_metric="RMSE",
            random_seed=seed + fold,
            verbose=300,
            early_stopping_rounds=200,
            allow_writing_files=False
        )
        
        model.fit(
            X_train,
            y_train,
            cat_features=cat_features,
            eval_set=(X_valid, y_valid),
            use_best_model=True
        )
        
        valid_preds = model.predict(X_valid)
        valid_preds = np.clip(valid_preds, 0, 1)
        
        seed_oof_preds[valid_idx] = valid_preds
        
        rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))
        mae = mean_absolute_error(y_valid, valid_preds)
        r2 = r2_score(y_valid, valid_preds)
        
        fold_rmse_scores.append(rmse)
        fold_mae_scores.append(mae)
        fold_r2_scores.append(r2)
        
        print("Fold RMSE:", rmse)
        print("Fold MAE :", mae)
        print("Fold R²  :", r2)
        
        fold_test_preds = model.predict(X_test)
        seed_test_preds += fold_test_preds / n_splits
    
    seed_oof_preds = np.clip(seed_oof_preds, 0, 1)
    seed_test_preds = np.clip(seed_test_preds, 0, 1)
    
    seed_rmse = np.sqrt(mean_squared_error(y, seed_oof_preds))
    seed_mae = mean_absolute_error(y, seed_oof_preds)
    seed_r2 = r2_score(y, seed_oof_preds)
    
    print("\nSeed summary:")
    print("Seed:", seed)
    print("OOF RMSE:", seed_rmse)
    print("OOF MAE :", seed_mae)
    print("OOF R²  :", seed_r2)
    print("Test pred std:", seed_test_preds.std())
    
    seed_score_rows.append({
        "seed": seed,
        "oof_rmse": seed_rmse,
        "oof_mae": seed_mae,
        "oof_r2": seed_r2,
        "test_pred_min": seed_test_preds.min(),
        "test_pred_max": seed_test_preds.max(),
        "test_pred_mean": seed_test_preds.mean(),
        "test_pred_std": seed_test_preds.std()
    })
    
    all_seed_oof_preds.append(seed_oof_preds)
    all_seed_test_preds.append(seed_test_preds)

# ============================================================
# 8. Average predictions across seeds
# ============================================================

ensemble_oof_preds = np.mean(all_seed_oof_preds, axis=0)
ensemble_test_preds = np.mean(all_seed_test_preds, axis=0)

ensemble_oof_preds = np.clip(ensemble_oof_preds, 0, 1)
ensemble_test_preds = np.clip(ensemble_test_preds, 0, 1)

ensemble_rmse = np.sqrt(mean_squared_error(y, ensemble_oof_preds))
ensemble_mae = mean_absolute_error(y, ensemble_oof_preds)
ensemble_r2 = r2_score(y, ensemble_oof_preds)

print("\n" + "#" * 80)
print("V014 Seed Ensemble Results")
print("#" * 80)
print("Ensemble OOF RMSE:", ensemble_rmse)
print("Ensemble OOF MAE :", ensemble_mae)
print("Ensemble OOF R²  :", ensemble_r2)

print("\nPrediction distribution:")
print("Prediction min:", ensemble_test_preds.min())
print("Prediction max:", ensemble_test_preds.max())
print("Prediction mean:", ensemble_test_preds.mean())
print("Prediction std:", ensemble_test_preds.std())

# ============================================================
# 9. Create submission
# ============================================================

sub_v014 = pd.DataFrame({
    "record_id": test_ids,
    "flood_risk_score": ensemble_test_preds
})

validate_submission(sub_v014)

os.makedirs("../submissions", exist_ok=True)

sub_v014.to_csv(
    "../submissions/sub_v014_catboost_alpha_pack_dropmissing_seed_ensemble.csv",
    index=False
)

print("\nSaved: ../submissions/sub_v014_catboost_alpha_pack_dropmissing_seed_ensemble.csv")

# ============================================================
# 10. Save comparison reports
# ============================================================

seed_scores_df = pd.DataFrame(seed_score_rows)

comparison = pd.DataFrame([
    {
        "version": "sub_v010_catboost_alpha_pack_dropmissing",
        "model": "Single CatBoost Alpha Pack Drop-Missing",
        "rows": 15518,
        "features": "35 Alpha Pack",
        "cv_rmse": 0.23180475883494558,
        "cv_mae": 0.17660570642311946,
        "cv_r2": 0.04023286825885215,
        "prediction_std": 0.044751843087494254,
        "public_score": 0.38176
    },
    {
        "version": "sub_v014_catboost_alpha_pack_dropmissing_seed_ensemble",
        "model": f"Seed Ensemble CatBoost, seeds={seeds}",
        "rows": len(train),
        "features": "35 Alpha Pack",
        "cv_rmse": ensemble_rmse,
        "cv_mae": ensemble_mae,
        "cv_r2": ensemble_r2,
        "prediction_std": ensemble_test_preds.std(),
        "public_score": None
    }
])

comparison["rmse_vs_v010"] = comparison["cv_rmse"] - comparison.loc[0, "cv_rmse"]
comparison["mae_vs_v010"] = comparison["cv_mae"] - comparison.loc[0, "cv_mae"]
comparison["r2_vs_v010"] = comparison["cv_r2"] - comparison.loc[0, "cv_r2"]
comparison["pred_std_vs_v010"] = comparison["prediction_std"] - comparison.loc[0, "prediction_std"]

os.makedirs("../reports", exist_ok=True)

seed_scores_df.to_csv(
    "../reports/v014_seed_scores.csv",
    index=False
)

comparison.to_csv(
    "../reports/v014_seed_ensemble_comparison.csv",
    index=False
)

print("\nSeed-level scores:")
display(seed_scores_df)

print("\nComparison with V010:")
display(comparison)

Train path: ../MLOpsedian/data/processed/train_dropmissing.csv
Test path: ../MLOpsedian/data/processed/test_dropmissing.csv
Train shape: (15518, 69)
Test shape: (5300, 68)
Train missing: 0
Test missing: 0
Duplicate train IDs: 0
Duplicate test IDs: 0
Train-only columns: {'flood_risk_score'}
Test-only columns: set()
Target valid: True

After feature engineering:
Train shape: (15518, 73)
Test shape: (5300, 72)
Train missing: 0
Test missing: 0

Feature check:
Number of Alpha Pack features: 35
Missing train features: []
Missing test features: []

Model data:
X shape: (15518, 35)
y shape: (15518,)
X_test shape: (5300, 35)
Missing in X: 0
Missing in X_test: 0
Feature columns match: True

Categorical features:
Number of categorical features: 10
['district', 'generation_date', 'reason_not_good_to_live', 'road_quality', 'landcover', 'place_name', 'water_supply', 'water_presence_flag', 'flood_occurrence_current_event', 'soil_type']

Target bin distribution:
flood_risk_score
0    1899
1    3455
2 

,seed,oof_rmse,oof_mae,oof_r2,test_pred_min,test_pred_max,test_pred_mean,test_pred_std
0,42,0.231808,0.176606,0.040235,0.367827,0.632363,0.479113,0.044752
1,202,0.231751,0.176810,0.040710,0.364201,0.642449,0.480106,0.044211
2,777,0.231717,0.176543,0.040992,0.362635,0.642047,0.477864,0.045169
3,2026,0.231919,0.176808,0.039317,0.361560,0.640691,0.478067,0.044315
4,9999,0.231840,0.176768,0.039971,0.364934,0.649443,0.478767,0.045063



Comparison with V010:


,version,model,rows,features,cv_rmse,cv_mae,cv_r2,prediction_std,public_score,rmse_vs_v010,mae_vs_v010,r2_vs_v010,pred_std_vs_v010
0,sub_v010_catboost_alpha_pack_dropmissing,Single CatBoost Alpha Pack Drop-Missing,15518,35 Alpha Pack,0.231805,0.176606,0.040233,0.044752,0.38176,0.000000,0.000000,0.00000,0.00000
1,sub_v014_catboost_alpha_pack_dropmissing_seed_...,"Seed Ensemble CatBoost, seeds=[42, 202, 777, 2...",15518,35 Alpha Pack,0.231572,0.176419,0.042192,0.044582,NaN,-0.000233,-0.000187,0.00196,-0.00017
